In [ ]:
# Notebook 3 — Jointures et agrégations
!pip install pyspark

In [6]:
from pyspark.sql import SparkSession
spark =  (SparkSession.builder
            .appName("TradeCorp ETL")
            .getOrCreate()
        )

ModuleNotFoundError: No module named 'pyspark'

In [ ]:

df_customers = spark.read.parquet("/home/jovyan/data/tmp/customers.parquet")
df_employees = spark.read.parquet("/home/jovyan/data/tmp/employees_enrich.parquet")
df_orders = spark.read.parquet("/home/jovyan/data/tmp/orders.parquet")
df_order_details = spark.read.parquet("/home/jovyan/data/tmp/order_details.parquet")
df_categories = spark.read.parquet("/home/jovyan/data/tmp/categories.parquet")
df_products = spark.read.parquet("/home/jovyan/data/tmp/products.parquet")
df_shippers = spark.read.parquet("/home/jovyan/data/tmp/shippers.parquet")
df_suppliers = spark.read.parquet("/home/jovyan/data/tmp/suppliers.parquet")



In [ ]:
# Q21 — Jointure orders + customers
# Joindre df_orders et df_customers sur customer_id. Garder uniquement : order_id, company_name, country,
# order_date, freight
df_customers_orders =  df_orders.join(df_customers, on="customer_id",how="inner").select("order_id","company_name","country","order_date","freight")
df_customers_orders.show(1)

In [ ]:
# Q22 — Jointure order_details + products
# Joindre df_order_details et df_products sur product_id. Ajouter les colonnes product_name, category_id,
# unit_price depuis products.
df_order_detailts_products = df_order_details.join(df_products.select("product_name","category_id","unit_price","product_id"), on="product_id",how="inner")
df_order_detailts_products.show(1)

In [ ]:
# Q23 — Jointure products + categories
# Joindre df_products et df_categories sur category_id pour enrichir chaque produit avec category_name et
# description.
df_products_categories = df_products.join(df_categories.select("category_id","category_name","description"), on="category_id", how="inner")
df_products_categories.show(1)

In [ ]:
# Q24 — DataFrame enrichi complet
# A - Réaliser une première jointure complète (order_details, orders, customers, products enrichi avec
# categories, employees, shippers) sans renommer aucune colonne. Lister ensuite les colonnes qui apparaissent
# en double grâce à Counter.

df_orders_enriched = df_orders\
    .join(df_order_details, on="order_id",how="inner")\
    .join(df_customers, on="customer_id",how="inner")\
    .join(df_products_categories, on="product_id", how="inner")\
    .join(df_employees, on="employee_id",how="inner")\
    .join(df_shippers,on="shipper_id",how="inner")


In [ ]:
# B - Pour chaque colonne identifiée en Q24a, la renommer dans sa table d'origine avant de refaire la jointure,
# en la préfixant selon la table (customer_country, employee_country, shipper_name...). Reconstruire ensuite
# df_orders_enriched avec ces tables renommées, puis vérifier qu'il ne reste plus aucun doublon

list_col = df_orders_enriched.columns
# df_list_col = spark.createDataFrame(list_col_dup, ["name"])
# df_list_col.groupBy("name").count().show()
 
col_dupli = []
for col in list_col:
    if list_col.count(col)> 1:
        col_dupli.append(col)
print(col_dupli)

df_all = {
    'customers':df_customers,
    'orders':df_orders,
    'product_categories':df_products_categories,
    'order_details':df_order_details,
    'shippers': df_shippers,
    'employees': df_employees
}
for name in df_all:
    df_customers_col = df_all[name].columns
    for col_customers in df_customers_col: 
        if col_customers in col_dupli:
            df_all[name] = df_all[name].withColumnRenamed(col_customers,name+'_'+ col_customers)


df_all["employees"].columns
            



In [ ]:
df_customers_renamed = df_all["customers"]
df_orders_renamed = df_all["orders"]
df_products_categories_renamed = df_all["product_categories"]
df_order_details_renamed = df_all["order_details"]
df_shippers_renamed = df_all["shippers"]
df_employees_renamed = df_all["employees"]

df_employees_renamed.columns


In [ ]:
df_orders_enriched = df_orders_renamed.join(df_order_details_renamed, on="order_id",how="inner")\
    .join(df_customers_renamed, on="customer_id",how="inner")\
    .join(df_products_categories_renamed, on="product_id", how="inner")\
    .join(df_employees_renamed, on="employee_id",how="inner")\
    .join(df_shippers_renamed,on="shipper_id",how="inner")


In [ ]:
df_orders_enriched.columns

In [ ]:
#  Q25 — CA par client
# Calculer le chiffre d'affaires total par client (company_name) depuis df_orders_enriched. Trier par CA
# décroissant. Afficher le top 10.

df_ca_client = df_orders_enriched \
    .groupBy("customer_company_name") \
    .agg(F.round(F.sum("sous_total"), 2).alias("ca_total")) \
    .orderBy(F.desc("ca_total"))

df_ca_client.show(10, truncate=False)

In [ ]:
# Q27 — CA mensuel
df_ca_mensuel = df_orders_enriched \
    .withColumn("mois_annee", F.date_trunc("month", "order_date")) \
    .groupBy("mois_annee") \
    .agg(F.round(F.sum("sous_total"), 2).alias("ca_total")) \
    .orderBy("mois_annee")

df_ca_mensuel.show()

In [ ]:
# Q28 — Performance par employé

df_perf_employe = df_orders_enriched \
    .withColumn("delai_livraison", F.datediff(F.col("shipped_date"), F.col("order_date"))) \
    .groupBy("full_name") \
    .agg(
        F.countDistinct("order_id").alias("nb_commandes"),
        F.round(F.sum("sous_total"), 2).alias("ca_total"),
        F.round(F.avg("delai_livraison"), 1).alias("delai_moyen_livraison_jours")
    ) \
    .orderBy(F.desc("ca_total"))

df_perf_employe.show()

In [ ]:
# Q29 — Window function : Rang des produits par CA dans chaque catégorie
from pyspark.sql.window import Window
window_category = Window.partitionBy("category_name").orderBy(F.desc("ca_produit"))

df_rang_produit = df_orders_enriched \
    .groupBy("category_name", "product_name") \
    .agg(F.round(F.sum("sous_total"), 2).alias("ca_produit")) \
    .withColumn("rang", F.dense_rank().over(window_category))

df_rang_produit.show()

In [ ]:
# Q30 — Window function : CA cumulé par mois

window_cumul = Window.orderBy("mois_annee").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_ca_cumule = df_ca_mensuel \
    .withColumn("ca_cumule", F.round(F.sum("ca_total").over(window_cumul), 2))

df_ca_cumule.show()

In [ ]:
# Q31 — Tri et limite
# Top 5 produits les plus vendus en quantité
df_top_produits_quantite = df_orders_enriched \
    .groupBy("product_name") \
    .agg(F.sum("quantite").alias("total_quantite")) \
    .orderBy(F.desc("total_quantite")) \
    .limit(5)

df_top_produits_quantite.show()

# Top 3 pays clients générant le plus de CA
df_top_pays_ca = df_orders_enriched \
    .groupBy("customer_country") \
    .agg(F.round(F.sum("sous_total"), 2).alias("ca_total")) \
    .orderBy(F.desc("ca_total")) \
    .limit(3)

df_top_pays_ca.show()

In [ ]:
# Q32 — Écriture finale en Parquet

df_orders_enriched.write \
    .mode("overwrite") \
    .parquet("/home/jovyan/data/output/orders_enriched.parquet")